In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb

path = r'C:\Users\abc\Desktop\ml\400.xlsx'
data = pd.read_excel(path)
print(data.shape)
print(data.columns.tolist())
print(data.head())

(1078, 4)
['Time (s)', 'H (0-360)', 'S (0-100)', 'V (0-100)']
   Time (s)  H (0-360)  S (0-100)  V (0-100)
0     0.012     161.73       4.81      75.99
1     0.071     162.77       4.78      75.97
2     0.141     162.45       4.82      75.98
3     0.209     153.86       4.22      72.76
4     0.272     152.19       5.57      73.19


In [7]:
# HSV 颜色还原可视化（可交互）
# 自动识别 H/S/V 列（支持不同命名）
cols = data.columns.tolist()
h_col = [c for c in cols if c.lower().startswith('h') or 'hue' in c.lower()][0]
s_col = [c for c in cols if c.lower().startswith('s') or 'sat' in c.lower()][0]
v_col = [c for c in cols if c.lower().startswith('v') or 'val' in c.lower() or 'value' in c.lower()][0]

h = data[h_col].values.astype(float)
s = data[s_col].values.astype(float)
v = data[v_col].values.astype(float)

# 归一化到 0-1 范围
if h.max() > 1:
    h = h / 360.0
if s.max() > 1:
    s = s / 100.0 if s.max() <= 100 else s / 255.0
if v.max() > 1:
    v = v / 100.0 if v.max() <= 100 else v / 255.0

# 组合为 HSV 并转 RGB
hsv = np.stack([h, s, v], axis=-1)
rgb = hsv_to_rgb(hsv)

n = len(rgb)
ncols = min(int(np.ceil(np.sqrt(n))), 30)

# 网格坐标
x = np.arange(n) % ncols
y = -(np.arange(n) // ncols)

# 颜色字符串
colors = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r, g, b in rgb]

# 时间列
time_col = [c for c in cols if 'time' in c.lower()][0]
time = data[time_col].values.astype(float)

# 悬停文本
hover_text = [
    f'时间: {time[i]:.3f}s  索引: {i}  H: {h[i]*360:.1f}  S: {s[i]*100:.1f}%  V: {v[i]*100:.1f}%  '
    f'RGB: ({int(rgb[i,0]*255)}, {int(rgb[i,1]*255)}, {int(rgb[i,2]*255)})'
    for i in range(n)
]

import plotly.graph_objects as go

fig = go.Figure(go.Scatter(
    x=x, y=y,
    mode='markers',
    marker=dict(
        size=30,
        color=colors,
        symbol='square',
        line=dict(width=1, color='LightSlateGrey')
    ),
    hovertext=hover_text,
    hoverinfo='text'
))

fig.update_layout(
    title=f'HSV 颜色可视化 (共 {n} 个颜色)',
    xaxis=dict(showgrid=True, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=True, zeroline=False, showticklabels=False, scaleanchor='x'),
    width=ncols*50,
    height=(n//ncols + 1)*50,
    hovermode='closest',
    margin=dict(l=20, r=20, t=40, b=20)
)
fig.show()

print(f'H: {h_col}, S: {s_col}, V: {v_col}, Time: {time_col}')


H: H (0-360), S: S (0-100), V: V (0-100), Time: Time (s)
